# Interpolation using standard lactation curves Tutorial

**ISLC** means *Interpolation using Standard Lactation Curves*. It estimates cumulative lactation yield from intermittent test-day records by using a standard lactation curve to fill gaps between observations and projects these to a predefined grid. 

In this tutorial we will create a fake dataset and apply the ISLC method to it. It will consist of the following parts:

1. Apply `ISLC_method` to one lactation to calculate a 305 day milk yield.
2. Apply `ISLC` to a dataframe containing many lactations.
3. Explore settings, including `max_dim` to get the result that works best for your situation.
4. Use your own data for a personalized lactation curve to make the estimation more accurate. 
5. Find more information with `help()` and the online documentation.
6. Compare the current ISLC implementation with the original paper of 
    Dr. Wilmink with the function `ISLC_original_method`.
7. Compare the available interpolation functions.

The examples use the package default lactation curve first, 
so you can run them without preparing a dataset to fit your own lactation curve.

If the lactationcurve package is not installed yet, run:

```bash
% pip install lactationcurve
```

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from lactationcurve.characteristics.interpolation_using_standard_lactation_curves import (
    CORR_MATRIX,
    STANDARD_CURVE,
    ISLC_method,
    ISLC_method_single_lactation,
    ISLC_original_method,
    STDs,
    interpolation_standard_lc,
    linear_interpd_all_to_grid,
    linear_interpd_closest_to_grid,
)

C:\Users\Meike van Leerdam\bovi\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Prepare example data set

`TestId` identifies a lactation, `DaysInMilk` is the days since calving at the day of measurement, 
and `MilkingYield` is the measured 24-hour yield.

In [2]:
records = pd.DataFrame(
    {
        "TestId": [101, 101, 101, 101, 202, 202, 202, 202],
        "DaysInMilk": [17, 83, 142, 231, 12, 67, 151, 276],
        "MilkingYield": [24.0, 31.0, 29.0, 22.0, 21.0, 29.0, 27.0, 18.0],
    }
)
records

,TestId,DaysInMilk,MilkingYield
0,101,17,24.0
1,101,83,31.0
2,101,142,29.0
3,101,231,22.0
4,202,12,21.0
5,202,67,29.0
6,202,151,27.0
7,202,276,18.0


## How ISLC works

The diagram below summarizes the main calculation. ISLC uses the standard lactation curve and the covariance structure between grid days to estimate missing yields before calculating cumulative lactation yield.


In [3]:
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis("off")

steps = [
    (0.4, "1. Observed\ntest days", "DIM + milk yield\n(e.g. 17, 63, 121)"),
    (3.4, "2. Standardize\nrecords", "Group by TestId\nand clean columns"),
    (6.4, "3. Fill the ISLC\ngrid", "Use standard curve\n+ covariance"),
    (9.4, "4. Estimate\nmissing yields", "Predicted values at\ngrid days"),
    (12.0, "5. Integrate\nthe lactation", "Cumulative yield\nthrough max_dim"),
]

colors = ["#D9EAF7", "#E8E3F3", "#DCEEDC", "#FFF0CC", "#F8D8D8"]
for (x, title, detail), color in zip(steps, colors):
    box = FancyBboxPatch(
        (x, 1.35),
        1.8,
        2.2,
        boxstyle="round,pad=0.08,rounding_size=0.08",
        facecolor=color,
        edgecolor="#444444",
        linewidth=1.2,
    )
    ax.add_patch(box)
    ax.text(x + 0.9, 2.85, title, ha="center", va="center", fontsize=11, weight="bold")
    ax.text(x + 0.9, 2.0, detail, ha="center", va="center", fontsize=9)

for start, end in zip(steps[:-1], steps[1:]):
    ax.add_patch(
        FancyArrowPatch(
            (start[0] + 1.85, 2.45),
            (end[0] - 0.05, 2.45),
            arrowstyle="-|>",
            mutation_scale=18,
            linewidth=1.5,
            color="#555555",
        )
    )

ax.text(
    7.0,
    0.55,
    "The fixed grid makes lactations comparable and supplies the inputs needed for "
    "the final yield calculation.",
    ha="center",
    va="center",
    fontsize=10,
    style="italic",
)
fig.tight_layout()
plt.show()

<Figure size 1400x450 with 1 Axes>

## 1. Apply ISLC to a single lactation

`ISLC_method` works on one lactation at a time. It needs a standard curve, correlation matrix, and standard deviations. The package provides these three things by default, but the lower-level function requires us to pass them explicitly.

In [4]:
# filter out one lactation for testing
one_lactation = records.loc[records["TestId"] == 101, ["DaysInMilk", "MilkingYield"]]

# Apply the calculation method to the single lactation data
single_result = ISLC_method_single_lactation(
    df=one_lactation,
    standard_lc=STANDARD_CURVE,
    correlation_matrix=CORR_MATRIX,
    std_per_grid_day=STDs,
)

# Print the estimated 305-day milk yield for the single lactation
print(f"Estimated 305-day yield: {single_result:.1f} kg")

Estimated 305-day yield: 8038.4 kg


## 2. Apply ISLC to a full dataframe

We can now run ISLC for one lactation, but we want to run it on all the lactations in the dataframe at once. We use `ISLC` as it is the convenient high-level function. It standardizes columns (so it does not get confused if *milkyield* is spelled like this or like *MilkYield* for instance), groups rows by `TestId`, runs `ISLC_method` for each lactation, and returns one result per lactation as a pandas dataframe. The unit of milk yield in the output is kg, unlike the test interval and centering date method, milk yields in pounds can not be directly used and need to be converted to kg before use. 

In [5]:
all_results = ISLC_method(records)
all_results

,TestId,LactationMilkYield
0,101,8038.421166
1,202,7492.492081


## 3. Settings you can change

You can provide alternative column names so the function works on your dataset, you can set a default ID when `TestId` is absent. If you want to make the method more tailored to your herd you can use a different set of standard curve, correlation matrix, and standard deviations. 

### Use your own column names 

In [6]:
# Simulate a dataset with different column names to test the aliasing functionality
alias_records = records.rename(
    columns={"TestId": "AnimalId", "DaysInMilk": "DIM", "MilkingYield": "Yield"}
)

# apply the ISLC method to the renamed columns
custom_names_result = ISLC_method(
    alias_records,
    days_in_milk_col="DIM",
    milking_yield_col="Yield",
    test_id_col="AnimalId",
)
# print the results of the ISLC method applied to the renamed columns
custom_names_result

,TestId,LactationMilkYield
0,101,8038.421166
1,202,7492.492081


### Calculate cumulative milk yield for lactations that are not 305 days long

The default is `max_dim=305`, which estimates a 305-day yield and ignores records beyond DIM 305. Set `max_dim=` to the desired lactation lenght (e.g. 270 days or 350 days), or use `max_dim="max"` to set the lactation lenght to be as long as the final measurement day. 

In [7]:
fixed_horizon = ISLC_method(records, max_dim=250)
observed_horizon = ISLC_method(records, max_dim="max")

print("250-DIM lactation length")
display(fixed_horizon)
print("Highest-observed-DIM lactation length")
display(observed_horizon)

250-DIM lactation length


,TestId,LactationMilkYield
0,101,6798.743913
1,202,6798.746547


Highest-observed-DIM lactation length


,TestId,LactationMilkYield
0,101,6378.920452
1,202,6931.799361


## 4. Fit your own standard lactation curve

A standard curve is most useful when it represents the same population or is very alike as the lactations you want to score. Pass a reference dataframe and set `fit_standard_lc_from_data=True`. The reference data should contain multiple representative lactations with `TestId`, `DaysInMilk`, and `MilkingYield` and it needs to have at least one value for each day in milk.

In [8]:
# Expand the example to three cows.
sparse_reference = pd.concat(
    [records, records.assign(TestId=303, MilkingYield=records["MilkingYield"] + 1.5)],
    ignore_index=True,
)


# Expand our example to a daily time series for each cow, using linear interpolation
# to fill in the missing days.
def expand_to_daily(group):
    daily_yield = (
        group.set_index("DaysInMilk")["MilkingYield"]
        .reindex(range(0, 306))
        .interpolate()
        .ffill()
        .bfill()
    )
    return pd.DataFrame(
        {
            "TestId": group["TestId"].iloc[0],
            "DaysInMilk": daily_yield.index,
            "MilkingYield": daily_yield.to_numpy(),
        }
    )


reference_records = (
    sparse_reference.groupby("TestId", group_keys=False)
    .apply(expand_to_daily)
    .reset_index(drop=True)
)

# Use the ISLC method whereby the standard lactation curve is fitted to the reference data,
# and then the fitted curve is used to estimate the 305-day yield for each cow
# in the original dataset.
fitted_assets_result = ISLC_method(
    records,
    fit_standard_lc_from_data=True,
    reference_df=reference_records,
)
fitted_assets_result

C:\Users\Meike van Leerdam\AppData\Local\Temp\ipykernel_656\3554899490.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(expand_to_daily)


,TestId,LactationMilkYield
0,101,7934.719157
1,202,7382.640651


The fitting option creates a Wilmink standard curve and estimates the correlation and standard-deviation of the milk recordings based on the reference records. For a production analysis, use a much larger reference population and keep the fitted assets together. This is why we call it a reference dataset. It is also possible to fit a lactation curve based on a subpopulation, e.g. all the parity 1 jersey cows in a population. To do this, filter your data into groups and give only the data from the group as a reference dataframe. 

## 5. Learn more with `help()` and documentation

Python's `help()` function displays the package docstring, including parameters, return values, and important notes. In VS Code or Jupyter, you can also use `?` after a function name.

In [9]:
help(ISLC_method)

Help on function ISLC_method in module lactationcurve.characteristics.interpolation_using_standard_lactation_curves:

ISLC_method(df: pandas.core.frame.DataFrame, days_in_milk_col: str | None = None, milking_yield_col: str | None = None, test_id_col: str | None = None, default_test_id: int = 0, standard_lc_305: pandas.core.series.Series | numpy.ndarray[tuple[typing.Any, ...], numpy.dtype[numpy.float64]] = STANDARD_CURVE, correlation_matrix: pandas.core.frame.DataFrame = CORR_MATRIX, std_per_grid_day: numpy.ndarray[tuple[typing.Any, ...], numpy.dtype[numpy.float64]] = STDs, max_dim: int | str = 305, fit_standard_lc_from_data: bool = False, reference_df: pandas.core.frame.DataFrame | None = None) -> pandas.core.frame.DataFrame
    Estimate cumulative lactation yield per TestId using ISLC.

    This variant applies :func:`ISLC_method` to standardized lactation records.
    By default it uses the package-provided standard curve, correlation matrix,
    and per-grid day standard deviations.

For the full API reference, see the [lactationcurve documentation](https://bovi-analytics.github.io/bovi/lactationcurve.html) and the [ISLC source module](https://github.com/Bovi-analytics/bovi/blob/main/packages/models/lactationcurve/src/lactationcurve/characteristics/interpolation_using_standard_lactation_curves.py).

## 6. ISLC original method: what is different?

The package also implements the original Wilmink-style fixed-grid method as `ISLC_original_method`.
This is the method as it was originally posted in the paper Wilmink, J. B. M. (1987).
Comparison of different methods of predicting 305-day milk yield using means
calculated from within-herd lactation curves. Livestock Production Science,
17, 1-17. 

Main differences:

- **Current `ISLC_method`:** This is the follow up of the original method that is a bit more flexible, puts more emphasize on measured values and can expand beyond 305 days/ 
- **`ISLC_original_method`:** works only with the fixed `GRID_DAYS` values, it is less flexible, cannot extend beyond 305 days, but is more robust and less computer intensive. 

In [10]:
original_results = ISLC_original_method(records)
comparison = all_results.merge(
    original_results,
    on="TestId",
    suffixes=("_ISLC", "_ISLC_original"),
)
comparison

,TestId,LactationMilkYield_ISLC,LactationMilkYield_ISLC_original
0,101,8038.421166,8078.427502
1,202,7492.492081,7525.635874


## 7. Compare interpolation methods

ISLC works under the hood by assigning milk yields to grid days and than apply the test interval method on it. The way to assign milk yields to predefined grid days can be varried. The default is to use the lactation curve for this. In most scenarios this is the most accurate way to do it. But if there is not enough data for a good lactation curve and the standard curve in the package does not work well for your population, linear interpolation can also be used. We have two versions of linear interpolation, one with extrapolation to each gridday and one that only interpolates intermediate values and estimates the values outside the grid with the ISLC method. 

- `interpolation_standard_lc`: recommended; uses the standard curve to guide the interpolation. (default)
- `linear_interpd_all_to_grid`: returns every grid day between DIM 0 and 305 using straight-line interpolation.
- `linear_interpd_closest_to_grid`: returns only grid days within the observed DIM range.

The default `ISLC_method` uses `interpolation_standard_lc`. 

The following cell will create a visual to illustrate the difference between the different interpolation values. 

In [11]:
standard_interpolated = interpolation_standard_lc(
    one_lactation,
    days_in_milk_col="DaysInMilk",
    milking_yield_col="MilkingYield",
    standard_lc=STANDARD_CURVE,
)
linear_all = linear_interpd_all_to_grid(one_lactation, "DaysInMilk", "MilkingYield")
linear_closest = linear_interpd_closest_to_grid(one_lactation, "DaysInMilk", "MilkingYield")

print("Standard-curve interpolation:")
display(standard_interpolated.head())
print("Linear interpolation on the full grid:")
display(linear_all.head())
print("Linear interpolation on grid days within the observed range:")
display(linear_closest.head() if linear_closest is not None else "No grid day is in range.")

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(
    one_lactation["DaysInMilk"],
    one_lactation["MilkingYield"],
    color="black",
    s=70,
    zorder=4,
    label="Observed test days",
)
ax.plot(
    standard_interpolated["GridDay"],
    standard_interpolated["MilkYieldInterp"],
    marker="o",
    linewidth=2.5,
    label="Standard-curve interpolation",
)
ax.plot(
    linear_all["GridDay"],
    linear_all["MilkYieldInterp"],
    marker="s",
    linestyle="--",
    alpha=0.8,
    label="Linear interpolation (full grid)",
)
if linear_closest is not None:
    ax.plot(
        linear_closest["GridDay"],
        linear_closest["MilkYieldInterp"],
        marker="^",
        linestyle=":",
        linewidth=2,
        label="Linear interpolation (observed range)",
    )
ax.set_title("How the interpolation methods fill the ISLC grid")
ax.set_xlabel("Days in milk")
ax.set_ylabel("Milking yield")
ax.set_xlim(left=0, right=305)
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

Standard-curve interpolation:


,DaysInMilk,MilkingYield,GridDay,MilkYieldInterp
0,17.0,24.0,30,27.548253
1,17.0,24.0,50,29.956537
2,17.0,24.0,70,30.794846
3,17.0,24.0,90,30.813478
4,17.0,24.0,110,30.169991


Linear interpolation on the full grid:


,GridDay,MilkYieldInterp
0,0,22.196970
1,10,23.257576
2,30,25.378788
3,50,27.500000
4,70,29.621212


Linear interpolation on grid days within the observed range:


,GridDay,MilkYieldInterp
0,30,25.378788
1,50,27.500000
2,70,29.621212
3,90,30.762712
4,110,30.084746


<Figure size 1100x600 with 1 Axes>

## What you learned

- ISLC fills gaps between measured test days and projects the results onto a fixed DIM grid.
- These known grid days are used to calculate the missing grid days using among others known covariance between test days. 
- Finally all known yields and the griddays are combined with the test interval method to calculate a final lactation length. 
- There are multiple ways to do the interpolation: Standard-curve interpolation uses the expected lactation shape, while linear interpolation uses only the measured yields.
- `ISLC_method` estimates cumulative yield for one or more lactations, and `max_dim` controls the lactation lenght.
- Custom standard curves and corresponding covariance and standard deviations can be fitted from a representative reference population.
- A reference population used for fitting must provide at least have a measurement for each day in milk, these may be from different animals. 